In [41]:
import glob
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

plt.style.use("ggplot")
plt.figure(figsize=(15, 10))

<Figure size 1500x1000 with 0 Axes>

<Figure size 1500x1000 with 0 Axes>

In [42]:
# checked for missing rows


all_files = glob.glob("raw_data/CRMLSSold*.csv")
all_dfs = []

for file in all_files:
    df_file = pd.read_csv(file, low_memory=False)
    all_dfs.append(df_file)

raw_master_df = pd.concat(all_dfs, ignore_index=True)

is_sfr = (raw_master_df["PropertyType"] == "Residential") & (raw_master_df["PropertySubType"] == "SingleFamilyResidence")
sfr_df = raw_master_df[is_sfr]

sfr_df = sfr_df.dropna(subset=["ClosePrice"])
sfr_df = sfr_df[sfr_df["ClosePrice"] <= 10_000_000]

features = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeArea"]
sfr_features_df = sfr_df[features]

print("--- Missing Value Counts ---")
sfr_features_df.isnull().sum()


--- Missing Value Counts ---


ClosePrice                  0
LivingArea                121
BedroomsTotal               0
BathroomsTotalInteger      37
LotSizeArea              4734
dtype: int64

In [43]:
#checked which cities had the most missing lot size area since it was the most important

missing_lot_mask = sfr_df["LotSizeArea"].isnull()
missing_lot_df = sfr_df[missing_lot_mask]


print("--- Top cities w/ missing lot sizes ---")

print(missing_lot_df["City"].value_counts().head(10))

print("\n")
print("--- ClosePrice summary ---")
print(missing_lot_df["ClosePrice"].describe())

--- Top cities w/ missing lot sizes ---
City
San Diego        1753
Escondido         248
Chula Vista       247
El Cajon          188
Oceanside         167
Poway             129
La Mesa           127
Spring Valley     119
La Jolla          111
Carlsbad          107
Name: count, dtype: int64


--- ClosePrice summary ---
count    4.734000e+03
mean     1.297796e+06
std      9.535948e+05
min      9.600000e+04
25%      7.900000e+05
50%      9.980000e+05
75%      1.435000e+06
max      9.500000e+06
Name: ClosePrice, dtype: float64


In [44]:
# imputed and created a flag collumn 

cleaned_df = sfr_df.copy()
cleaned_df["Missing_LotSizeArea"] = cleaned_df["LotSizeArea"].isnull().astype(int)

global_lotSize_median = cleaned_df["LotSizeArea"].median()

city_medians = cleaned_df.groupby("City")["LotSizeArea"].median()
# print(city_medians)

def fill_with_city_median(row):
    if pd.isnull(row["LotSizeArea"]):
        city = row["City"]
        return city_medians.get(city, global_lotSize_median)
    else:
        return row["LotSizeArea"]

cleaned_df["LotSizeArea"] = cleaned_df.apply(fill_with_city_median, axis=1)

cleaned_df["LivingArea"] = cleaned_df["LivingArea"].fillna(cleaned_df["LivingArea"].median())
cleaned_df["BathroomsTotalInteger"] = cleaned_df["BathroomsTotalInteger"].fillna(cleaned_df["BathroomsTotalInteger"].median())
cleaned_df["LotSizeArea"] = cleaned_df["LotSizeArea"].fillna(cleaned_df["LotSizeArea"].median())

In [45]:
# one hot encoded the only categorical variable which is the city 

ohe = OneHotEncoder(drop='first', sparse_output=False)

city_matrix = ohe.fit_transform(cleaned_df[['City']])

city_feature_names = ohe.get_feature_names_out(['City'])

city_encoded_df = pd.DataFrame(
    city_matrix, 
    columns = city_feature_names, 
    index = cleaned_df.index,
    dtype = int
)

final_features = [
    "ClosePrice", 
    "LivingArea", 
    "BedroomsTotal", 
    "BathroomsTotalInteger", 
    "LotSizeArea", 
    "Missing_LotSizeArea"
]

final_df = pd.concat([cleaned_df[final_features], city_encoded_df], axis=1)

print(f"Dataset shape: {final_df.shape}")
# final_df.head()
# final_df[final_df['City_Acampo'] == 1].head(5)

Dataset shape: (277266, 1112)


In [46]:
# I normalized in case I use Lasso / Ridge later 
numeric_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeArea"]

final_df = final_df.copy()

final_df[numeric_cols] = ((final_df[numeric_cols] - final_df[numeric_cols].mean()) / final_df[numeric_cols].std())

# final_df[numeric_cols].describe()

final_df.to_csv("raw_data/cleaned_data.csv", index=False)

In [47]:
split_idx = int(len(final_df) * 0.96)

X = final_df.drop(columns=["ClosePrice"])
y = final_df["ClosePrice"]

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

Train features shape: (266175, 1111)
Test features shape: (11091, 1111)
